> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

# 02. ROS graph와 turtlesim 도구 / 60분

**선행:** 01의 Node·Topic·Message

**이 노트북의 결과:** graph에서 publisher/subscriber 연결을 읽고 같은 질문을 ROS CLI와 로컬 GUI로 확인합니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

프로그램이 실행 중이라는 사실만으로 원하는 데이터가 전달되는 것은 아닙니다. 이 실습은 graph에서 누가 어떤 이름·타입으로 연결되었는지 읽고, 키보드 명령과 거북이의 실제 위치 변화를 연결하는 관찰 연습입니다.

### 전체 과정에서 내 코드가 하는 일

```text
topic 요약 TODO
  ↓
node·topic·endpoint 확인
  ↓
B 키보드 입력
  ↓
cmd_vel 요청
  ↓
pose 관찰
  ↓
연결 설명
```

graph 요약을 구현하고 로컬 공동 turtlesim에서 입력 터미널의 초점을 확인합니다. 원격 개인 Domain에서는 결정적 모형을 사용하며 다른 Domain의 로컬 메시지를 기다리지 않습니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** publisher와 subscriber가 보이면 메시지가 도착했다고 말할 수 있나요?

**설명:** 연결 후보가 있다는 증거이며 실제 수신과 다릅니다. echo·위치 변화 같은 별도 근거를 확인합니다. cmd_vel은 요청한 속도, pose는 상태이므로 숫자나 단위가 같은 것이 아닙니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**ROS 준비 — 노트북의 Python 모의 실험과 실제 ROS 실행 구분:** 아래 Python 셀은 새 fixture와 loader를 직접 만들므로 앞 노트북의 커널 변수·TODO 결과가 필요 없습니다. 실제 node는 **같은 개인 컨테이너의 JupyterLab Terminal** 에서 실행합니다. Ollama는 필요 없습니다.

[01_ros_basics.ipynb의 3-1~3-3절](01_ros_basics.ipynb)에 workspace 빌드와 `source`의 설명이 있으며, 필요한 절차는 다음과 같습니다. 빌드는 `ros_ws/src`의 패키지를 `Data/build/08.ROS/install`에 설치·연결합니다. 첫 사용 또는 install 파일이 없을 때, 패키지 설정·실행 이름이 바뀌었을 때 다음을 실행하고 성공을 확인합니다. 같은 자료 폴더·호환 이미지의 빌드 결과가 남아 있으면 재부팅만을 이유로 다시 빌드할 필요는 없습니다.

```bash
bash
cd /workspace
bash 08.ROS/scripts/build_workspace.sh
```

**새 터미널 A·B·C마다** 아래를 적용합니다. 이 설정은 현재 터미널에만 적용되며 다른 터미널·Python 커널로 전파되지 않습니다. 21은 로컬 예시이고 원격 개인 실습은 22 등 배정값을 씁니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 pkg executables nepes_ros_lab
```

`status_talker`, `image_monitor`, `safe_turtle_controller` 등 실행 이름이 보이면 준비된 것입니다. 기본 ROS는 첫 source, 수업 패키지는 두 번째 source가 찾게 합니다. `Package not found`는 build 성공·현재 install source를 확인합니다. 같은 실습의 모든 터미널은 같은 Domain 번호를 사용합니다. 로컬 GUI를 함께 관찰할 때는 두 수강생이 로컬 컨테이너의 화면을 함께 보고, 원격 개인 Domain에서 로컬 토픽이 보일 것으로 기대하지 않습니다. 이전 실습의 source/controller·카메라를 종료한 뒤 다음 실행을 시작합니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[01_ros_basics.ipynb의 1·3절](01_ros_basics.ipynb)에 Node·Topic·Message와 실제 실행이 있습니다. 필요한 내용은 **node=실행 참가자, topic=데이터 통로의 이름, message=필드·타입 계약**, publisher가 발행하고 subscriber의 callback이 받는다는 것입니다. 같은 Domain·topic·type과 호환 QoS가 필요합니다. `cmd_vel`의 Twist는 요청 속도, `pose`는 관측 위치입니다. 아래 `MiniBus()`와 `GRAPH`를 새로 만들므로 01의 `bus`나 실행 중인 talker는 필요 없습니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–4절의 수업 핵심 경로** → [3일차 슬라이드 51쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s047) · [배포 PDF 51쪽](../슬라이드/3일차.pdf#page=51)에서 57쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 입력 토픽·처리 결과·출력 명령을 확인하고 종료합니다.
- **단원 마지막 연결 질문:** 입력 노드를 중단하고 마지막 관측 시각, STOP, 복구 후 새 관측을 확인합니다. 운영자가 어느 로그부터 볼지 설명합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

[ROS 터미널 운영표](../08.ROS/실행안내.md)를 사용하세요. 모의 입력은 이동·정지를, 실제 카메라는 인식·승인 또는 거부 이유를 확인합니다. 

08의 MediaPipe confidence=null 이동은 기본 controller에서 거부됩니다. 입력 중단 실험은 실행안내의 source·controller 분리 명령으로 진행합니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 시간 지도

환경 5분 → graph 12분 → CLI 13분 → TODO 15분 → turtlesim/local·headless 10분 → 종료 5분.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "08.ROS").is_dir():
            return candidate
    raise RuntimeError("08.ROS를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "08.ROS"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('rclpy', 'cv2', 'cv_bridge')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. Graph는 실행 중 상태다

Graph는 source 폴더 구조가 아니라 현재 살아 있는 node와 endpoint의 관계입니다.

```text
/camera --/image_raw--> /vision --/gesture--> /controller
```

Publisher만 있으면 소비자가 없고 subscriber만 있으면 입력을 기다립니다. 양쪽이 있어도 type이나 QoS가 호환되지 않으면 전달되지 않을 수 있습니다.

| 질문 | CLI |
|---|---|
| node 목록/상세 | `ros2 node list`, `ros2 node info /vision` |
| topic과 type | `ros2 topic list -t` |
| endpoint와 QoS | `ros2 topic info /image_raw --verbose` |
| 내용/빈도 | `ros2 topic echo ... --once`, `ros2 topic hz ...` |


In [ ]:
# 구분: 수업 핵심 셀
from common.mini_ros import MiniBus
bus = MiniBus()
for name in ("/camera", "/vision", "/controller"):
    bus.add_node(name)
bus.create_publisher("/camera", "/image_raw")
bus.create_subscription("/vision", "/image_raw", lambda message: None)
bus.create_publisher("/vision", "/gesture")
bus.create_subscription("/controller", "/gesture", lambda message: None)
bus.create_publisher("/controller", "/unused")
GRAPH = bus.graph()
print(json.dumps(GRAPH, ensure_ascii=False, indent=2))


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 2. TODO — Topic 연결 요약

[exercises/graph_contract.py](exercises/graph_contract.py)의 `describe_topic`을 구현합니다. Publisher/subscriber 이름을 중복 제거·정렬하고 양쪽이 하나 이상일 때만 `connected=True`로 둡니다. 없는 topic은 빈 endpoint 목록으로 설명합니다. 이름이 보인다는 사실과 실제 data가 온다는 사실을 구분하세요.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `describe_topic(graph,topic)`는 `graph={'topics':{'/x':{'publishers':['a','a'],'subscribers':['b']}}}`, topic='/x'에서 `{'topic':'/x','publishers':['a'],'subscribers':['b'],'connected':True}`를 반환합니다. 없는 topic은 두 목록이 []이고 connected=False입니다.
**작성 흐름 힌트:** graph와 topics가 mapping인지 검사 → 선택 topic 항목 조회 → endpoint가 문자열 list인지 검사 → set으로 중복 제거하고 sorted → 양쪽이 비어 있지 않은지 bool로 반환. 입력 snapshot을 수정하지 않습니다.
**설명할 지점:** “연결 가능성이 보인다는 뜻이지 데이터 수신이나 QoS 호환까지 증명한 것은 아닙니다.”
**누락·오류 처리와 통과 기준:** `graph`가 Mapping이 아니면 `TypeError`; `graph['topics']`나 조회된 topic 항목이 Mapping이 아니면 `ValueError`입니다. topic이 없으면 빈 항목 `{}`으로 처리하고, `publishers`·`subscribers` 필드가 없으면 각각 `[]`입니다. 해당 필드가 list가 아니거나 원소에 문자열 아닌 값이 있으면 `ValueError`로 거부합니다. `collections.abc.Mapping`을 직접 import해 검사할 수 있습니다. 반환 dict의 key는 정확히 topic/publishers/subscribers/connected이며 입력을 수정하지 않습니다. 아래 검증의 연결·한쪽만 존재·topic 없음·중복/정렬 사례가 모두 PASS여야 합니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
student = load_file("student_ros_graph", "exercises/graph_contract.py")
image = student.describe_topic(GRAPH, "/image_raw")
unused = student.describe_topic(GRAPH, "/unused")
missing = student.describe_topic(GRAPH, "/missing")
assert image == {"topic": "/image_raw", "publishers": ["/camera"], "subscribers": ["/vision"], "connected": True}
assert unused["publishers"] == ["/controller"] and not unused["connected"]
assert missing == {"topic": "/missing", "publishers": [], "subscribers": [], "connected": False}
messy = {
    "topics": {
        "/mixed": {
            "publishers": ["/z_source", "/a_source", "/z_source"],
            "subscribers": ["/b_sink", "/a_sink", "/b_sink"],
        },
        "/subscriber_only": {"publishers": [], "subscribers": ["/sink"]},
    }
}
assert student.describe_topic(messy, "/mixed") == {
    "topic": "/mixed",
    "publishers": ["/a_source", "/z_source"],
    "subscribers": ["/a_sink", "/b_sink"],
    "connected": True,
}
assert student.describe_topic(messy, "/subscriber_only") == {
    "topic": "/subscriber_only", "publishers": [], "subscribers": ["/sink"], "connected": False,
}
print("PASS: 연결, 한쪽 endpoint, topic 없음, 중복 제거와 결정적 정렬")


 > **구분: 수업 핵심**

## 참고 정답

강사가 안내하는 시점에 아래 참고 정답과 해설을 확인합니다. [solutions/graph_contract.py](solutions/graph_contract.py)에 입력 검사·계산·반환이 직접 구현되어 있으며, 아래 접힌 Markdown에도 같은 코드가 있습니다. 정답 검증과 학습자 TODO 검증은 별도입니다.


In [ ]:
# 구분: 수업 핵심 셀
reference = load_file("reference_ros_graph", "solutions/graph_contract.py")
assert reference.describe_topic(GRAPH, "/image_raw")["connected"]
assert not reference.describe_topic(GRAPH, "/unused")["connected"]
print("PASS: 참고 정답 검증")


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/graph_contract.py](solutions/graph_contract.py)의 `describe_topic`입니다. graph['topics']에서 지정 topic을 찾고 publisher·subscriber 목록의 타입을 검사합니다.

set으로 중복을 제거하고 sorted로 순서를 정하면 실행 순서가 달라도 같은 연결 목록을 비교할 수 있습니다. publisher와 subscriber가 모두 있어야 connected=True입니다. 없는 topic은 빈 목록 두 개와 False입니다.

graph snapshot에 연결이 있다는 사실은 실제 영상 내용이 맞거나 새 메시지가 도착한다는 뜻은 아닙니다. 실제 환경에서는 node list→topic type→echo/hz 순으로 증거를 더 확인합니다.

**확인 질문:** publisher 하나가 있고 subscriber가 0개이면 발행 프로세스가 정상 실행돼도 왜 수신 실습은 완료가 아닌가요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/graph_contract.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: topic의 발행자·구독자 이름을 중복 제거하고 정렬합니다.
양쪽 목록이 모두 있을 때 connected=True이며 실제 수신 성공을 뜻하지는 않습니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
from typing import Any, Mapping

def describe_topic(graph: Mapping[str, Any], topic: str) -> dict[str, Any]:
    """graph snapshot에서 특정 topic의 연결 관계를 정규화한다."""
    if not isinstance(graph, Mapping):
        raise TypeError("graph는 mapping이어야 합니다")
    topics = graph.get("topics")
    if not isinstance(topics, Mapping):
        raise ValueError("graph['topics'] mapping이 필요합니다")
    entry = topics.get(topic, {})
    if not isinstance(entry, Mapping):
        raise ValueError("topic 항목은 mapping이어야 합니다")
    publishers = entry.get("publishers", [])
    subscribers = entry.get("subscribers", [])
    if not isinstance(publishers, list) or not all(isinstance(v, str) for v in publishers):
        raise ValueError("publishers는 문자열 list여야 합니다")
    if not isinstance(subscribers, list) or not all(isinstance(v, str) for v in subscribers):
        raise ValueError("subscribers는 문자열 list여야 합니다")
    # 이름을 정규화할 뿐 실제 수신이나 QoS 호환까지 증명하지 않습니다.
    return {
        "topic": topic,
        "publishers": sorted(set(publishers)),
        "subscribers": sorted(set(subscribers)),
        "connected": bool(publishers and subscribers),
    }
```

</details>
<!-- todo-review:answer:end -->


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 3. turtlesim 실제 경로

<!-- procedure:lesson-27-c23005:start -->
![터미널별로 제어와 관찰 나누기](../교재그림/절차/lesson-27.svg)

A·B·C는 함께 실행합니다. 순서 번호는 준비·확인 순서이며 C가 제어를 대신하는 것은 아닙니다.
<!-- procedure:lesson-27-c23005:end -->


로컬 Jetson에서는 turtlesim_node, turtle_teleop_key, rqt_graph를 서로 다른 터미널에서 실행합니다. USB·GUI 공동 실습은 로컬 컨테이너에서 두 수강생이 함께 진행합니다. 아래 관찰 터미널도 같은 로컬 Domain 21 예시입니다. 원격 개인 Domain 22에서는 앞의 Python graph 실습을 독립 실행하며 로컬 turtlesim 메시지를 기다리지 않습니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 이 공동 GUI 실습의 로컬 배정값
# 로컬 터미널 A
ros2 run turtlesim turtlesim_node
```

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 이 공동 GUI 실습의 로컬 배정값
ros2 run turtlesim turtle_teleop_key
```

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 이 공동 GUI 실습의 로컬 배정값
# 로컬 컨테이너의 관찰 터미널 C
mkdir -p /workspace/Data/outputs/08.ROS
ros2 node list
ros2 topic info /turtle1/cmd_vel --verbose
ros2 topic echo /turtle1/pose --once | tee /workspace/Data/outputs/08.ROS/02-turtle-pose.txt

# 로컬 GUI만
rqt_graph
```

cmd_vel은 요청한 commanded state이고 pose는 관측된 state입니다. 원격 사용자는 현장의 로컬 사용자가 turtlesim을 실행한 경우에만 같은 ROS graph를 관찰하며, 개인 Domain ID라면 결정적 Python graph fixture로 확인합니다.

### 직접 조작하고 관찰하기

1. A는 거북이 창을 유지하고, **B의 JupyterLab Terminal 내부를 클릭** 하여 키보드 입력 초점을 둡니다. 거북이 그림을 클릭하는 것으로는 B에 키 입력이 전달되지 않습니다.
2. B가 출력한 키 안내를 확인하고 방향키로 전진·후진·회전을 시도합니다. 위·아래는 선속도, 왼쪽·오른쪽은 회전 방향을 바꾸는 입력입니다. 서로 다른 키를 차례로 눌러 위치와 방향의 변화를 구분합니다.
3. C에서 위치·방향을 관찰하고, 키를 놓은 뒤에도 계속 움직이는지 잠시 뒤 정지하는지 기록합니다. `cmd_vel`은 요청이고 `pose`는 실제 상태이므로 둘을 같은 값으로 해석하지 않습니다.
4. 움직이지 않으면 B의 초점 → A/B 프로세스 → 동일한 Domain → 토픽 연결 순으로 확인합니다. GUI가 뜨는 것만으로 키 입력까지 성공한 것은 아닙니다. 종료는 실행 중인 각 터미널에서 Ctrl+C로 합니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 4. 종료·재실행

각 장시간 node 터미널에서 `Ctrl+C`를 누르고 `ros2 node list`를 다시 확인합니다. Source를 고쳤다면 colcon build → 새 shell의 install setup source → run 순서입니다. Graph 그림만 보고 끝내지 말고 endpoint 수, type, 실제 message를 근거로 연결을 설명합니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** topic 요약 검증·실제 graph·로컬 turtlesim 확인

[3일차 슬라이드 51쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s047) · [배포 PDF 51쪽](../슬라이드/3일차.pdf#page=51)로 돌아가 **51–57쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

## 자습 — Namespace와 remap을 실제 이름으로 추적하기 / 약 40분

Namespace는 여러 robot이 같은 짧은 이름을 사용해도 최종 ROS 이름이 겹치지 않게 앞 경로를 붙입니다. Remap은 source를 수정하지 않고 실행할 때 연결 이름을 바꿉니다.

- 상대 이름 status는 namespace /robot_a 안에서 /robot_a/status가 됩니다.
- /factory/status처럼 슬래시로 시작한 절대 이름은 namespace만으로 바뀌지 않습니다.
- 절대 원본 이름을 상대 목표 status로 remap하고 namespace를 주면 최종 이름은 /robot_a/status입니다.

아래 Python fixture는 이 수업에서 쓰는 세 규칙만 표현합니다. ROS 2의 전체 이름 해석기를 대신하지 않습니다.

진행 순서는 fixture 읽기 → 최종 이름 예측 → 검증 → 실제 headless ROS 명령 → Data 기록입니다. 로컬과 원격 모두 GUI 없이 실행할 수 있습니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
STUDY_NAME_CASES = [
    {
        "id": "relative_in_namespace",
        "namespace": "/robot_a",
        "source_name": "status",
        "remap_target": None,
    },
    {
        "id": "absolute_ignores_namespace",
        "namespace": "/robot_a",
        "source_name": "/factory/status",
        "remap_target": None,
    },
    {
        "id": "absolute_remapped_to_relative",
        "namespace": "/robot_a",
        "source_name": "/factory/status",
        "remap_target": "status",
    },
]
EXPECTED_RESOLVED_NAMES = {
    "relative_in_namespace": "/robot_a/status",
    "absolute_ignores_namespace": "/factory/status",
    "absolute_remapped_to_relative": "/robot_a/status",
}
print(json.dumps(STUDY_NAME_CASES, ensure_ascii=False, indent=2))


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### TODO — 최종 topic 이름 예측

다음 셀의 None 세 곳을 슬래시로 시작하는 최종 topic 이름으로 바꾸세요.

1. Remap target이 있으면 source name 대신 target부터 봅니다.
2. 선택된 이름이 이미 슬래시로 시작하면 절대 이름이므로 그대로 둡니다.
3. 상대 이름이면 namespace 뒤에 붙입니다.

문자열의 슬래시 개수와 소문자를 그대로 확인하세요.

**입력과 편집 위치:** 세 사례는 모두 namespace `/robot_a`이며 source가 각각 `status`, `/factory/status`, `/factory/status`입니다. 세 번째만 remap target=`status`이고 앞의 둘은 remap이 없습니다. 아래 `MY_RESOLVED_NAMES`의 key를 유지하고 값 세 개만 **절대 이름 문자열** 로 바꿉니다. 이름 해석 함수를 새로 만드는 과제가 아닙니다. 결과는 `/`로 시작하며 remap→절대/상대 판정→namespace 결합 순서에 맞아야 합니다. 수정 셀과 검증을 다시 실행해 세 개가 모두 PASS인 뒤 `02-namespace-remap.json`의 completed=True를 확인합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
MY_RESOLVED_NAMES = {
    "relative_in_namespace": None,       # TODO: "/..." 형식
    "absolute_ignores_namespace": None,  # TODO
    "absolute_remapped_to_relative": None,  # TODO
}
print("현재 답:", MY_RESOLVED_NAMES)


In [ ]:
# 구분: 수업 후 복습·자습 셀
missing = [name for name, value in MY_RESOLVED_NAMES.items() if value is None]
name_study_passed = False
if missing:
    print("TODO가 남았습니다:", missing)
else:
    assert set(MY_RESOLVED_NAMES) == set(EXPECTED_RESOLVED_NAMES), "case ID를 바꾸지 마세요."
    for name, expected in EXPECTED_RESOLVED_NAMES.items():
        actual = MY_RESOLVED_NAMES[name]
        assert isinstance(actual, str) and actual.startswith("/"), f"{name}: 절대 이름 문자열이 필요합니다."
        assert actual == expected, f"{name}: {actual!r}; namespace/remap 순서를 다시 확인하세요."
    name_study_passed = True
    print("PASS: 상대 이름, 절대 이름, remap 뒤 최종 이름을 구분했습니다.")


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### 실제 ROS에서 같은 규칙 확인

이 노트북 앞의 「ROS 준비」에 있는 build 절차로 패키지를 준비한 뒤, 터미널 A·B **각각** 에서 `bash` → `source /opt/ros/humble/setup.bash` → `source /workspace/Data/build/08.ROS/install/setup.bash` → `export ROS_DOMAIN_ID=21`을 실행합니다. 원격은 21 대신 본인 배정값을 두 터미널에 동일하게 적용합니다. 두 명령 모두 GUI가 없으므로 원격 사용자도 실행할 수 있습니다.

터미널 A:

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 본인 배정값 또는 공동 관찰의 로컬 배정값
ros2 run nepes_ros_lab status_talker --ros-args -r __ns:=/robot_a -r /factory/status:=status
```

터미널 B:

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 본인 배정값 또는 공동 관찰의 로컬 배정값
ros2 topic list -t
ros2 topic info /robot_a/status --verbose
ros2 topic echo /robot_a/status --once
```

예상 결과는 /robot_a/status에 publisher 한 개가 보이고 JSON 문자열 한 건이 출력되는 것입니다. 끝나면 터미널 A에서 Ctrl+C를 누르고 node list로 종료를 확인합니다.


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### 예상 결과와 해설

- relative_in_namespace → /robot_a/status
- absolute_ignores_namespace → /factory/status
- absolute_remapped_to_relative → /robot_a/status

Namespace만 주었을 때 절대 이름이 자동으로 바뀐다고 예상했다면 두 번째 사례를 다시 확인하세요. 실제 연결 이름은 항상 ros2 topic list와 topic info로 확인합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
name_study_output = ROOT / "Data/outputs/08.ROS/02-namespace-remap.json"
name_study_output.parent.mkdir(parents=True, exist_ok=True)
name_study_record = {
    "exercise": "namespace_remap",
    "answers": MY_RESOLVED_NAMES,
    "expected": EXPECTED_RESOLVED_NAMES,
    "completed": MY_RESOLVED_NAMES == EXPECTED_RESOLVED_NAMES,
}
name_study_output.write_text(
    json.dumps(name_study_record, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
assert name_study_output.is_file()
print("saved:", name_study_output)
print("completed:", name_study_record["completed"])
